# Australian Greyhound Racing — Staking & Strategy Backtester

**Prerequisite:** Run `greyhound_eda.ipynb` first to produce `greyhound_races.parquet`.

This notebook backtests five strategies on the full historical dataset:

| # | Strategy | Selection | Staking |
|---|----------|-----------|--------|
| 1 | Lay favourite | BSP favourite every race | Flat $10 liability |
| 2 | Back favourite | BSP favourite every race | Flat $10 stake |
| 3 | Back shortener | Favourite that shortened ≥15% morning→BSP | Flat $10 stake |
| 4 | Back value | Runner where BSP > calibrated fair price | Flat $10 stake |
| 5 | Kelly lay | Lay favourite when implied prob > true prob | Kelly-scaled liability |

All strategies use **BSP execution** (Betfair Starting Price) with **5% commission** on winning bets.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

OUTPUT_DIR  = './greyhound_output'
COMMISSION  = 0.05
FLAT_STAKE  = 10.0   # $ per bet (back bets) or $ liability (lay bets)
KELLY_FRAC  = 0.25   # fractional Kelly to limit variance
STARTING_BANK = 1_000.0

races = pd.read_parquet(os.path.join(OUTPUT_DIR, 'greyhound_races.parquet'))
cal   = pd.read_csv(os.path.join(OUTPUT_DIR, 'bsp_calibration.csv'))

races['event_dt'] = pd.to_datetime(races['event_dt'])
races = races.sort_values('event_dt').reset_index(drop=True)

print(f'Loaded {len(races):,} races  ({races["event_dt"].min().date()} → {races["event_dt"].max().date()})')
print(f'With BSP: {races["has_bsp"].sum():,}  ({races["has_bsp"].mean()*100:.1f}%)')

## Calibration — Build True Probability Model

Fit a simple isotonic regression over the BSP calibration curve to convert any BSP into an empirical true win probability.

In [ ]:
from sklearn.isotonic import IsotonicRegression

# Load runner-level data for calibration fit
runners = pd.read_parquet(os.path.join(OUTPUT_DIR, 'greyhound_runners.parquet'))
runners = runners[runners['BSP'].notna() & (runners['BSP'] > 1.01)].copy()
runners['implied_prob'] = 1.0 / runners['BSP']

iso = IsotonicRegression(out_of_bounds='clip', increasing=True)
iso.fit(runners['implied_prob'], runners['WIN_LOSE'])

def bsp_to_true_prob(bsp_series):
    """Convert BSP to calibrated true win probability via isotonic regression."""
    implied = 1.0 / bsp_series.clip(lower=1.01)
    return pd.Series(iso.predict(implied), index=bsp_series.index)

# Validate
test_bsps = [1.5, 2.0, 3.0, 5.0, 10.0, 20.0]
print('Calibration model — BSP → true probability:')
print(f'{"BSP":>6s}  {"Implied":>8s}  {"True prob":>10s}  {"Edge":>8s}')
for b in test_bsps:
    implied = 1/b
    true_p  = iso.predict([[implied]])[0]
    edge    = true_p - implied
    print(f'{b:>6.1f}  {implied*100:>7.1f}%   {true_p*100:>9.1f}%  {edge*100:>+7.2f}pp')

## Backtesting Framework

In [ ]:
def backtest_back(selections, starting_bank=STARTING_BANK, commission=COMMISSION):
    """
    Backtest a series of back bets at BSP.

    selections: DataFrame with columns:
        event_dt, bsp, won (1/0), stake

    Returns per-bet P&L DataFrame and summary dict.
    """
    sel = selections.copy().reset_index(drop=True)
    sel['pnl'] = np.where(
        sel['won'] == 1,
        sel['stake'] * (sel['bsp'] - 1) * (1 - commission),
        -sel['stake']
    )
    sel['bank'] = starting_bank + sel['pnl'].cumsum()
    sel['drawdown'] = sel['bank'] - sel['bank'].cummax()
    return sel


def backtest_lay(selections, starting_bank=STARTING_BANK, commission=COMMISSION):
    """
    Backtest a series of lay bets at BSP.

    selections: DataFrame with columns:
        event_dt, bsp, won (1/0 — 1 = runner WON = lay LOSES), liability

    Lay mechanics:
        back_stake = liability / (bsp - 1)
        if runner doesn't win: profit = back_stake * (1 - commission)
        if runner wins:        loss   = liability
    """
    sel = selections.copy().reset_index(drop=True)
    back_stake = sel['liability'] / (sel['bsp'] - 1)
    sel['pnl'] = np.where(
        sel['won'] == 0,
        back_stake * (1 - commission),
        -sel['liability']
    )
    sel['bank'] = starting_bank + sel['pnl'].cumsum()
    sel['drawdown'] = sel['bank'] - sel['bank'].cummax()
    return sel


def summary(result, name, stake_col='stake'):
    n       = len(result)
    wins    = result['won'].sum() if 'won' in result.columns else (result['pnl'] > 0).sum()
    total_staked = result[stake_col].sum()
    total_pnl    = result['pnl'].sum()
    roi          = total_pnl / total_staked * 100 if total_staked > 0 else 0
    max_dd       = result['drawdown'].min()
    final_bank   = result['bank'].iloc[-1]
    monthly_pnl  = result.set_index('event_dt')['pnl'].resample('ME').sum()
    sharpe       = monthly_pnl.mean() / monthly_pnl.std() * np.sqrt(12) if monthly_pnl.std() > 0 else 0
    return {
        'Strategy':      name,
        'Bets':          n,
        'Win rate':      f'{wins/n*100:.1f}%',
        'Total P&L':     f'${total_pnl:+,.0f}',
        'ROI':           f'{roi:+.2f}%',
        'Max drawdown':  f'${max_dd:,.0f}',
        'Final bank':    f'${final_bank:,.0f}',
        'Ann. Sharpe':   f'{sharpe:.2f}',
    }

## Strategy 1 — Lay the Favourite (Flat Liability)

In [ ]:
s1 = races[races['fav_bsp'].notna()][['event_dt','fav_bsp','fav_won']].copy()
s1 = s1.rename(columns={'fav_bsp':'bsp','fav_won':'won'})
s1['liability'] = FLAT_STAKE
s1['stake']     = s1['liability'] / (s1['bsp'] - 1)
r1 = backtest_lay(s1)
sum1 = summary(r1, 'Lay Favourite (flat $10 liability)', stake_col='liability')
print(sum1)

## Strategy 2 — Back the Favourite (Flat Stake)

In [ ]:
s2 = races[races['fav_bsp'].notna()][['event_dt','fav_bsp','fav_won']].copy()
s2 = s2.rename(columns={'fav_bsp':'bsp','fav_won':'won'})
s2['stake'] = FLAT_STAKE
r2 = backtest_back(s2)
sum2 = summary(r2, 'Back Favourite (flat $10)', stake_col='stake')
print(sum2)

## Strategy 3 — Back Morning Shorteners

Back the favourite only when its price shortened ≥15% from morning market to BSP (informed money signal).

In [ ]:
s3 = races[races['fav_bsp'].notna() & races['fav_morningwap'].notna()].copy()
s3['drift'] = (s3['fav_morningwap'] - s3['fav_bsp']) / s3['fav_morningwap']
s3 = s3[s3['drift'] >= 0.15][['event_dt','fav_bsp','fav_won']].copy()
s3 = s3.rename(columns={'fav_bsp':'bsp','fav_won':'won'})
s3['stake'] = FLAT_STAKE
r3 = backtest_back(s3)
sum3 = summary(r3, 'Back Shortener ≥15% (flat $10)', stake_col='stake')
print(f'Selections: {len(s3):,}')
print(sum3)

## Strategy 4 — Back Value (Calibrated Edge)

Back the favourite when the calibrated true probability exceeds the BSP implied probability by ≥2pp (positive expected value).

In [ ]:
s4 = races[races['fav_bsp'].notna()].copy()
s4['true_prob']    = bsp_to_true_prob(s4['fav_bsp'])
s4['implied_prob'] = 1.0 / s4['fav_bsp']
s4['edge']         = s4['true_prob'] - s4['implied_prob']
s4 = s4[s4['edge'] >= 0.02][['event_dt','fav_bsp','fav_won','edge']].copy()
s4 = s4.rename(columns={'fav_bsp':'bsp','fav_won':'won'})
s4['stake'] = FLAT_STAKE
r4 = backtest_back(s4)
sum4 = summary(r4, 'Back Value ≥2pp edge (flat $10)', stake_col='stake')
print(f'Selections: {len(s4):,}')
print(sum4)

## Strategy 5 — Kelly Lay (Calibrated)

Lay the favourite when the BSP implied probability exceeds the calibrated true probability (favourite is overbet).
Stake via fractional Kelly on the liability.

In [ ]:
s5 = races[races['fav_bsp'].notna()].copy()
s5['true_prob']    = bsp_to_true_prob(s5['fav_bsp'])
s5['implied_prob'] = 1.0 / s5['fav_bsp']
s5['lay_edge']     = s5['implied_prob'] - s5['true_prob']   # positive = favourite overbet

# Kelly fraction for laying:
# f = (p_lay_win * (bsp-1) - p_lay_lose) / (bsp-1)
# where p_lay_win = 1 - true_prob, p_lay_lose = true_prob
s5 = s5[s5['lay_edge'] > 0].copy()
q = 1 - s5['true_prob']   # prob lay wins
b = s5['fav_bsp'] - 1     # lay odds - 1 (exposure multiple)
s5['kelly_f']  = ((q * b - s5['true_prob']) / b).clip(0, 0.5)
s5['kelly_f'] *= KELLY_FRAC   # fractional Kelly

# Bank-scaled liability (start with STARTING_BANK, update as bank changes)
bank = STARTING_BANK
liabilities = []
for _, row in s5.iterrows():
    liab = bank * row['kelly_f']
    liabilities.append(liab)
    back_stake = liab / (row['fav_bsp'] - 1)
    if row['fav_won'] == 0:
        bank += back_stake * (1 - COMMISSION)
    else:
        bank -= liab
    bank = max(bank, 0)

s5 = s5[['event_dt','fav_bsp','fav_won']].copy()
s5 = s5.rename(columns={'fav_bsp':'bsp','fav_won':'won'})
s5['liability'] = liabilities
s5['stake']     = s5['liability'] / (s5['bsp'] - 1)
r5 = backtest_lay(s5)
sum5 = summary(r5, f'Kelly Lay (frac={KELLY_FRAC})', stake_col='liability')
print(f'Selections: {len(s5):,}')
print(sum5)

## Results Summary

In [ ]:
summaries = [sum1, sum2, sum3, sum4, sum5]
summary_df = pd.DataFrame(summaries)
print('\n=== Strategy Comparison ===')
print(summary_df.to_string(index=False))

## Equity Curves

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9))
fig.suptitle('Strategy Backtests — Australian Greyhound BSP', fontsize=13)

results = [
    (r1, 'Lay Favourite',       '#e74c3c'),
    (r2, 'Back Favourite',      '#3498db'),
    (r3, 'Back Shortener ≥15%', '#2ecc71'),
    (r4, 'Back Value ≥2pp',     '#f39c12'),
    (r5, 'Kelly Lay',           '#9b59b6'),
]

ax1 = axes[0]
for r, label, color in results:
    ax1.plot(r['event_dt'], r['bank'], label=label, color=color, linewidth=1.5, alpha=0.85)
ax1.axhline(STARTING_BANK, color='black', linewidth=0.8, linestyle='--')
ax1.set_ylabel('Bank ($)')
ax1.set_title('Bank Value Over Time')
ax1.legend(fontsize=8)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

ax2 = axes[1]
for r, label, color in results:
    ax2.plot(r['event_dt'], r['drawdown'], label=label, color=color, linewidth=1.2, alpha=0.75)
ax2.fill_between(results[0][0]['event_dt'], results[0][0]['drawdown'], 0,
                  alpha=0.05, color=results[0][2])
ax2.set_ylabel('Drawdown ($)')
ax2.set_title('Drawdown Over Time')
ax2.legend(fontsize=8)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.show()

## Monthly P&L Breakdown

In [ ]:
fig, axes = plt.subplots(len(results), 1, figsize=(14, 3*len(results)), sharex=True)
fig.suptitle('Monthly P&L by Strategy', fontsize=13)

for ax, (r, label, color) in zip(axes, results):
    monthly = r.set_index('event_dt')['pnl'].resample('ME').sum()
    bar_colors = ['#2ecc71' if p >= 0 else '#e74c3c' for p in monthly]
    ax.bar(monthly.index, monthly.values, color=bar_colors, alpha=0.85, width=20)
    ax.axhline(0, color='black', linewidth=0.6)
    ax.set_ylabel('P&L ($)')
    ax.set_title(label, fontsize=9)
    pos_months = (monthly > 0).sum()
    ax.text(0.01, 0.95, f'{pos_months}/{len(monthly)} profitable months',
             transform=ax.transAxes, fontsize=7, va='top')

plt.tight_layout()
plt.show()

## Walk-Forward Validation

Train the calibration model on 70% of data, test on the remaining 30%.
Prevents look-ahead bias in the value/Kelly strategies.

In [ ]:
from sklearn.isotonic import IsotonicRegression

# Chronological split
split_date = races['event_dt'].quantile(0.70)
print(f'Train: up to {split_date.date()}   Test: from {split_date.date()}')

train_runners = runners[runners['EVENT_DT'] < split_date]
test_races    = races[races['event_dt'] >= split_date].copy()

# Refit calibration on training data only
iso_wf = IsotonicRegression(out_of_bounds='clip', increasing=True)
tr = train_runners[train_runners['BSP'].notna()].copy()
iso_wf.fit(1.0 / tr['BSP'].clip(lower=1.01), tr['WIN_LOSE'])

def bsp_to_true_prob_wf(bsp_series):
    implied = 1.0 / bsp_series.clip(lower=1.01)
    return pd.Series(iso_wf.predict(implied), index=bsp_series.index)

# Re-run Strategies 3, 4, 5 on test set only
wf_results = {}

# Strategy 3 — shortener
s3t = test_races[test_races['fav_bsp'].notna() & test_races['fav_morningwap'].notna()].copy()
s3t['drift'] = (s3t['fav_morningwap'] - s3t['fav_bsp']) / s3t['fav_morningwap']
s3t = s3t[s3t['drift'] >= 0.15].rename(columns={'fav_bsp':'bsp','fav_won':'won'})
s3t['stake'] = FLAT_STAKE
wf_results['Back Shortener (OOS)'] = backtest_back(s3t)

# Strategy 4 — value
s4t = test_races[test_races['fav_bsp'].notna()].copy()
s4t['true_prob'] = bsp_to_true_prob_wf(s4t['fav_bsp'])
s4t['edge']      = s4t['true_prob'] - 1.0 / s4t['fav_bsp']
s4t = s4t[s4t['edge'] >= 0.02].rename(columns={'fav_bsp':'bsp','fav_won':'won'})
s4t['stake'] = FLAT_STAKE
wf_results['Back Value (OOS)'] = backtest_back(s4t)

print('\nOut-of-sample results:')
for name, res in wf_results.items():
    sc = 'stake' if 'stake' in res.columns else 'liability'
    print(summary(res, name, stake_col=sc))

## ROI by BSP Odds Bucket

In [ ]:
# Show ROI per odds bucket for back-favourite strategy (most data)
roi_df = r2.copy()
roi_df['odds_bucket'] = pd.cut(roi_df['bsp'],
                                bins=[1, 2, 3, 4, 6, 10, 20, 100],
                                labels=['1-2', '2-3', '3-4', '4-6', '6-10', '10-20', '20+'])

bucket_roi = roi_df.groupby('odds_bucket', observed=True).apply(
    lambda d: pd.Series({
        'n':        len(d),
        'win_rate': d['won'].mean() * 100,
        'total_pnl': d['pnl'].sum(),
        'total_staked': d['stake'].sum(),
        'roi': d['pnl'].sum() / d['stake'].sum() * 100
    })
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Back Favourite — ROI by Odds Bucket', fontsize=12)

colors_roi = ['#2ecc71' if r > 0 else '#e74c3c' for r in bucket_roi['roi']]
axes[0].bar(bucket_roi['odds_bucket'], bucket_roi['roi'], color=colors_roi, alpha=0.85)
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_xlabel('BSP odds range')
axes[0].set_ylabel('ROI (%)')
axes[0].set_title('ROI by Odds Bucket')

axes[1].bar(bucket_roi['odds_bucket'], bucket_roi['win_rate'], color='steelblue', alpha=0.85)
axes[1].set_xlabel('BSP odds range')
axes[1].set_ylabel('Win rate (%)')
axes[1].set_title('Win Rate by Odds Bucket')

plt.tight_layout()
plt.show()

print(bucket_roi.assign(
    win_rate=lambda d: d['win_rate'].round(1),
    roi=lambda d: d['roi'].round(2),
    total_pnl=lambda d: d['total_pnl'].round(0)
).to_string(index=False))

## Next Steps

Based on the backtest results, the most promising directions are:

1. **Feature engineering** — combine drift + volume + trap + grade into a single selection model (logistic regression or XGBoost)
2. **Venue-specific calibration** — some tracks may have persistent favourite biases
3. **Grade filtering** — if Juvenile/Maiden races show different calibration, build separate models
4. **Staking refinement** — if any strategy shows +EV, switch from flat to Kelly; use 25% fractional Kelly to manage variance
5. **RL agent** — use the real BSP data to train the `BettingEnv` agent on historical greyhound markets rather than synthetic odds